# Full experiment — scale testing — configured by one file

> **Full experiment.** Shows a **welcome** screen, then the **consent**
> questionnaire (it sets the anonymous `participant_id`), then the experiment,
> then a **thank-you** screen. Every saved file is named with that participant id.


This notebook runs a **scale test** and saves the ratings. On each screen the
participant hears **one stimulus** (a single play button) and rates it on
every scale stacked below — each scale is either a row of **Likert-style
buttons** or a **slider**, as configured. This is the classic setup for
semantic-differential / attribute-rating experiments.

Like the other whispy experiments it is **config-driven**: the whole thing is
described in [`configs/scale_testing.yml`](../configs/scale_testing.yml); only
the shared look comes from [`configs/design.yml`](../configs/design.yml). You
should not need to edit any Python below.

That one file is read by three consumers:

| block in the YAML | used by | configures |
|---|---|---|
| `SoundDevice:` | the audio handler | which WAV plays for each stimulus id |
| `ui:` / `questions:` | the ScaleTest window | window size, wording, and the rating scales |
| `trial:` / `experiment:` | this notebook | the prompt and which stimuli are rated (blocks/sections) |

If you haven't installed the project yet, run this from the repo root:
`pip install -e .`

## Welcome the participant

Run this first. It greets the participant with a **welcome screen** before
anything else; its message lives in [`configs/welcome.yml`](../configs/welcome.yml)
(markdown supported), so you can reword it without editing the Python. The cell
blocks until the participant clicks **Continue**.

In [ ]:
from pathlib import Path
from whispy.ui import InfoWindow
from whispy.utils import read_config

# The welcome message lives in configs/welcome.yml (markdown supported); reword
# it there without editing this cell.
welcome = read_config(Path('..') / 'configs' / 'welcome.yml')['ui']
InfoWindow(welcome['message'], fullscreen=welcome.get('fullscreen', True), blocking=True)

## Consent

Next, record consent: the cell below shows the consent questionnaire
([`configs/questionnaires/questionnaire_consent.yml`](../configs/questionnaires/questionnaire_consent.yml)),
which records consent and builds an **anonymous participant id** from the
`pid_1..pid_4` answers. The id is kept in the `participant_id` variable and used
to name every result file saved in this notebook, so all of this participant's
data shares one id.

In [ ]:
from pathlib import Path
from whispy.ui import Questionnaire
from whispy.utils import participant_id_from_consent, save_results

consent_config = Path('..') / 'configs' / 'questionnaires' / 'questionnaire_consent.yml'
consent = Questionnaire(questionnaire=consent_config, blocking=True)
consent_results = consent.get_results()
consent.close()

# Anonymous participant id (e.g. 'HPo1'); names every result file below.
participant_id = participant_id_from_consent(consent_results)
print('participant id:', participant_id)
save_results(consent_results, 'consent', participant_id=participant_id)

In [ ]:
import whispy
import pandas as pd
from pathlib import Path
from whispy.interfaces import SoundDevice
from whispy.ui import ScaleTest
from whispy.utils import read_config

config_path = Path('..') / 'configs' / 'scale_testing.yml'   # <- the one file you edit
stimuli_dir = Path('demo_stimuli/scale_testing')             # <- folder with your WAVs

cfg = read_config(config_path)
handler = SoundDevice(config_path, stimuli_dir, loop=False)   # reads SoundDevice:
print('available stimulus ids:', list(handler.stimuli.keys()))

## See your scale test at a glance

Open [`configs/scale_testing.yml`](../configs/scale_testing.yml) to edit every
setting (it has inline comments). The cell below prints the current values and
the questions, so you can review the design without leaving the notebook.

In [ ]:
print('ui        :', cfg['ui'])
print('trial     :', cfg['trial'])
print('experiment:', cfg['experiment'])
print()
print('questions (each is one scale on every screen):')
for i, q in enumerate(cfg['questions'], start=1):
    rng = q.get('scale_range', [1, 5])
    print(f"  {i}. {q.get('id', f'q{i}')}: {q.get('prompt')}"
          f"  [{q.get('interaction_method', 'buttons')}, {rng[0]}..{rng[1]}]")

## Using your own stimuli

The WAVs in `examples/demo_stimuli/scale_testing/` are **only a demo** (run
`python examples/demo_stimuli/scale_testing/generate_scale_testing_stimuli.py`
to regenerate them). For a real experiment you swap in your own files — you
only touch the config and `stimuli_dir`, never the Python.

1. **Point `stimuli_dir`** (in the setup cell) at the folder holding your WAVs.
2. **Map ids to files** under `SoundDevice:` in
   [`configs/scale_testing.yml`](../configs/scale_testing.yml), then list them
   under `experiment:` in the section's `test:` list:
   ```yaml
   SoundDevice:
     violin_dry: { file: violin_dry.wav }
     violin_hall: { file: violin_hall.wav }
   experiment:
     - block:
         - block_name: my block
         - section:
             section_name: violins
             test: [violin_dry, violin_hall]
   ```
3. **Design the scales** under `questions:` — per question choose
   `interaction_method` (`buttons` or `slider`), the `scale_range`, and
   optional `labels` (two entries = endpoints; for buttons, one per step =
   a label under each button).

**Safety constraints** (checked when the handler loads in the setup cell — a
violation raises a `ValueError`):

- **No clipping** — every sample must satisfy `|amplitude| < 1` (normalise with headroom, e.g. ~0.7).
- **One sampling rate** — every file must share the same sampling rate.
- **All ids must exist** — every id used under `experiment:` must be defined under `SoundDevice:`.

## Build the screens

`whispy.ExperimentScheduler` randomizes the `experiment:` block
(blocks/sections/stimuli, exactly like MUSHRA). Scale testing then presents
**one stimulus per screen**, so each scheduler screen (which carries a
section's stimuli as a list) is expanded into consecutive single-stimulus
screens, and the progress counter is recomputed over the expanded list.

In [ ]:
scheduler = whispy.ExperimentScheduler(experiment=cfg['experiment'])

# If per_stimulus_questions are defined, use those; otherwise use global questions.
# Pass them on the screen dict so ScaleTest can use stimulus-specific scales.
per_stim_q = cfg.get('per_stimulus_questions')

screens = []
for section_screen in scheduler:
    for stimulus in list(section_screen['test']):
        screen = {
            'stimulus': stimulus,
            'task': cfg['trial']['task'],
            'block': section_screen['block'],
            'section': section_screen['section'],
            'block_name': section_screen['block_name'],
            'section_name': section_screen['section_name'],
        }
        # Wire up per-stimulus questions if defined
        if per_stim_q is not None and isinstance(per_stim_q, dict):
            screen['questions'] = per_stim_q.get(stimulus, cfg.get('questions'))
        screens.append(screen)

for i, screen in enumerate(screens, start=1):
    screen['trial_id'] = i
    # drives the "Trial X of Y" progress bar (ui.show_progress)
    screen['progress'] = {'current': i, 'total': len(screens)}

print(f'{len(screens)} screens (one stimulus each):')
for screen in screens:
    print(f"  {screen['trial_id']:>2}. {screen['stimulus']}")

## Run it

The loop below presents every screen, reusing **one window** for the whole
experiment (the first screen opens it; later screens swap their content into
the same window via `parent=host`, so it stays fullscreen with no reload). The
window is closed in a `finally` block when the run finishes.

On each screen: press **Space** (or click **Play**) to hear the stimulus,
answer every scale (click a button, or drag/click a slider), then press
**Enter** to submit. Submitting requires the stimulus to have been played and
every scale to be answered.

In [ ]:
host = None
results = None

try:
    for screen in screens:
        scale_test = ScaleTest(screen=screen, stimuli_handler=handler,
                               scale_test_config=config_path,
                               blocking=True, parent=host)
        if host is None:
            host = scale_test    # first screen owns the shared window
        results = scale_test.get_results(results)
finally:
    if host is not None:
        host.close()

results

## Thank the participant

The experiment is over. Run this cell to show the participant a **thank-you
screen**; its message lives in [`configs/thanks.yml`](../configs/thanks.yml)
(markdown supported), so you can reword it without editing the Python.

In [ ]:
from whispy.ui import InfoWindow

# The thank-you message lives in configs/thanks.yml (markdown supported); reword
# it there without editing this cell.
thanks = read_config(Path('..') / 'configs' / 'thanks.yml')['ui']
InfoWindow(thanks['message'], fullscreen=thanks.get('fullscreen', True), blocking=True)

## Results

`results` is **long-form**: one row per question per stimulus (`stimulus`,
`question`, `answer`, `scale_min`, `scale_max`, `rt`, ...). The cell below
pivots it into a stimulus × question table of the given answers.

In [ ]:
print(results.pivot_table(index='stimulus', columns='question',
                          values='answer', aggfunc='mean').round(1).to_string())

## Save the results

Save the results to a CSV in a `results/` folder (created next to this
notebook). The file name always carries a **timestamp**. If a
`participant_id` is in scope (set by the consent block above) it is included
(`<name>_<id>_<timestamp>.csv`); otherwise an iterating fallback number is
used (`<name>_<NNN>_<timestamp>.csv`). Existing files are never overwritten.

In [ ]:
from whispy.utils import save_results

# `participant_id` is picked up automatically from the consent block above;
# otherwise it is None and the file name uses an iterating fallback number.
participant_id = globals().get('participant_id')
results_path = save_results(results, 'scale_testing', participant_id=participant_id)
print('saved results to', results_path.resolve())